# 🧩 Mini-Lab: Fake AI Client Tests

**Module: Tests That Catch Real Bugs** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Understand** what a test double is and why it replaces a real AI client in tests
2. **Apply** dependency injection to make functions testable without network access
3. **Verify** that retry logic and error-handling paths work correctly using fake clients that simulate failures
4. **Identify** how output-parsing tests can use fake responses to cover malformed JSON edge cases

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Test Double | An injected fake backend that can return malformed JSON or time out on demand, so tests reliably exercise error paths without calling real external services. |
| Dependency Injection | Pass the backend in, callers choose which model client to use, while your pipeline code stays unchanged and testable. |
| Bounded Retry | Retry transient errors only, and cap attempts, total wall-clock time, and cost or rate-limit budget so failures end fast, predictably, and safely. |
| Output Parsing | Output parsing is turning a model’s response into reliable, structured data (like JSON fields) your code can safely use,while detecting mistakes, missing info, or prompt-injection tricks in the output. |


## The Problem: Testing Code That Calls an LLM

Recall from the testability map: the LLM call itself is stochastic, but the **code around it** — retries, error handling, JSON parsing — is deterministic. The challenge is: how do you test that code without actually calling an API?

The answer: **inject a fake AI client** that returns exactly what you need — valid JSON, malformed JSON, or even raises an exception — so you can test every path reliably, offline, and for free.

In [1]:
!pip install -q pytest ipytest


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\molak\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import ipytest
ipytest.autoconfig()

print("ipytest ready.")

ipytest ready.


## Step 1 — The Hard-Coded Version (Not Testable)

Here's a typical first attempt at writing an extraction function. The AI client is created **inside** the function, so there's no way to swap it out in tests.

In [3]:
import json


# ❌ Hard-coded dependency — cannot test without a real API key
def extract_contact_hardcoded(text: str) -> dict:
    """This function creates its own client internally."""
    # from openai import OpenAI
    # client = OpenAI()  ← hard-coded, needs network + API key
    # response = client.chat.completions.create(...)
    # return json.loads(response.choices[0].message.content)
    raise NotImplementedError("Can't test this without a real API key!")


print("Hard-coded version defined (intentionally broken for illustration).")

Hard-coded version defined (intentionally broken for illustration).


## Step 2 — Dependency Injection Makes It Testable

**Dependency injection** simply means: pass the AI client **as a parameter**. The function doesn't care whether it gets a real OpenAI client or a fake — it just calls `.chat.completions.create()` on whatever object it receives.

We'll also add **bounded retry** logic (retry up to N times on failure) and **output parsing** (parse JSON from the response, handling malformed output).

In [4]:
import json
import time


def extract_contact(text: str, client, max_retries: int = 2) -> dict:
    """
    Extract contact info from text using an AI client.
    
    Args:
        text: Raw text containing contact information.
        client: Any object with a .chat.completions.create() method.
        max_retries: Number of retries on failure (bounded retry).
    
    Returns:
        Parsed dict with contact fields.
    
    Raises:
        RuntimeError: If all retries are exhausted.
        ValueError: If the response is not valid JSON.
    """
    messages = [
        {"role": "system", "content": "Extract name and email as JSON."},
        {"role": "user", "content": text},
    ]

    last_error = None
    for attempt in range(max_retries + 1):
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=messages,
            )
            raw = response.choices[0].message.content
            return json.loads(raw)  # output parsing
        except json.JSONDecodeError as e:
            raise ValueError(f"Malformed JSON from LLM: {e}") from e
        except Exception as e:
            last_error = e
            if attempt < max_retries:
                time.sleep(0)  # no real delay in tests

    raise RuntimeError(
        f"All {max_retries + 1} attempts failed. Last error: {last_error}"
    )


print("extract_contact() defined with dependency injection, bounded retry, and output parsing.")

extract_contact() defined with dependency injection, bounded retry, and output parsing.


Notice what changed: `client` is now a **parameter**. The function never imports or creates an OpenAI client. This single change makes the entire function testable offline.

## Step 3 — Building a Fake AI Client (Test Double)

A **test double** is a stand-in object that mimics the interface of the real dependency. Our fake client needs one method — `.chat.completions.create()` — and we control exactly what it returns (or whether it raises an error).

We'll build it with simple classes — no mocking library needed.

In [5]:
class FakeMessage:
    def __init__(self, content: str):
        self.content = content

class FakeChoice:
    def __init__(self, content: str):
        self.message = FakeMessage(content)

class FakeResponse:
    def __init__(self, content: str):
        self.choices = [FakeChoice(content)]


class FakeAIClient:
    """Test double for an OpenAI-style client."""

    def __init__(self, responses=None, error=None):
        """
        Args:
            responses: List of raw strings to return in sequence.
            error: Exception to raise on every call (simulates API failure).
        """
        self.responses = list(responses) if responses else []
        self.error = error
        self.call_count = 0
        self.chat = self  # so client.chat.completions.create() works
        self.completions = self

    def create(self, **kwargs):
        self.call_count += 1
        if self.error:
            raise self.error
        return FakeResponse(self.responses.pop(0))


print("FakeAIClient defined — supports controlled responses and simulated errors.")

FakeAIClient defined — supports controlled responses and simulated errors.


The fake mimics the OpenAI client's interface: `client.chat.completions.create()`. But instead of hitting a network, it returns whatever string we configure — or raises an exception we choose.

## Step 4 — Testing the Happy Path

First, let's confirm that when the fake client returns valid JSON, our function parses it correctly.

In [6]:
%%ipytest -v

import pytest


def test_happy_path_returns_parsed_dict():
    """Valid JSON response → parsed dict."""
    fake = FakeAIClient(
        responses=['{"name": "Alice", "email": "alice@example.com"}']
    )
    result = extract_contact("Contact Alice at alice@example.com", client=fake)

    assert result == {"name": "Alice", "email": "alice@example.com"}
    assert fake.call_count == 1  # called exactly once, no retries needed

======================================= test session starts =======================================
platform win32 -- Python 3.12.10, pytest-9.1.1, pluggy-1.6.0
rootdir: C:\Users\molak\MJ_Projects\19_atomic_ai_eng_course\1_ai_engineering_track\output\content\labs\mini-fake-ai-client-tests
plugins: anyio-4.9.0, asyncio-1.4.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 1 item

t_f30a8b00641c476a9facae09eb66c421.py 

.

                                                      [100%]



======================================== 1 passed in 0.09s ========================================


No network call. No API key. No cost. The test ran instantly and verified both the return value and that the client was called exactly once.

## Step 5 — Testing Malformed JSON (Output Parsing Failure)

What happens when the LLM returns something that isn't valid JSON? Our function should raise a `ValueError`. Let's prove it.

In [7]:
%%ipytest -v


def test_malformed_json_raises_value_error():
    """Non-JSON response → ValueError."""
    fake = FakeAIClient(responses=["Sorry, I can't extract that."])

    with pytest.raises(ValueError, match="Malformed JSON"):
        extract_contact("some text", client=fake)

======================================= test session starts =======================================
platform win32 -- Python 3.12.10, pytest-9.1.1, pluggy-1.6.0
rootdir: C:\Users\molak\MJ_Projects\19_atomic_ai_eng_course\1_ai_engineering_track\output\content\labs\mini-fake-ai-client-tests
plugins: anyio-4.9.0, asyncio-1.4.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 1 item

t_f30a8b00641c476a9facae09eb66c421.py 

.

                                                      [100%]



======================================== 1 passed in 0.04s ========================================


Without a fake client, testing this path would require somehow getting a real LLM to produce invalid JSON — unreliable and expensive. With the fake, we control the exact response.

## Step 6 — Testing Bounded Retry on API Failures

Now the most valuable test: does the retry logic actually work? We simulate an API that always times out.

In [8]:
%%ipytest -v


def test_retries_exhausted_raises_runtime_error():
    """All retries fail → RuntimeError with attempt count."""
    fake = FakeAIClient(error=TimeoutError("Connection timed out"))

    with pytest.raises(RuntimeError, match="All 3 attempts failed"):
        extract_contact("some text", client=fake, max_retries=2)

    # 1 initial attempt + 2 retries = 3 total calls
    assert fake.call_count == 3


def test_single_attempt_when_no_retries():
    """max_retries=0 means exactly one attempt."""
    fake = FakeAIClient(error=ConnectionError("Network down"))

    with pytest.raises(RuntimeError, match="All 1 attempts failed"):
        extract_contact("some text", client=fake, max_retries=0)

    assert fake.call_count == 1

======================================= test session starts =======================================
platform win32 -- Python 3.12.10, pytest-9.1.1, pluggy-1.6.0
rootdir: C:\Users\molak\MJ_Projects\19_atomic_ai_eng_course\1_ai_engineering_track\output\content\labs\mini-fake-ai-client-tests
plugins: anyio-4.9.0, asyncio-1.4.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 2 items

t_f30a8b00641c476a9facae09eb66c421.py 

.

.

                                                     [100%]



======================================== 2 passed in 0.04s ========================================


These tests verify:

- The retry loop runs exactly the right number of times (bounded, not infinite).
- The final error message includes useful context.
- No real network was touched — the entire test suite runs in milliseconds.

## The Pattern at a Glance

```
Production code:   extract_contact(text, client=OpenAI())
Test code:         extract_contact(text, client=FakeAIClient(...))
```

| Technique | What it does | Why it matters |
|-----------|-------------|----------------|
| **Test Double** | `FakeAIClient` stands in for the real OpenAI client | No network, no cost, deterministic |
| **Dependency Injection** | `client` is a parameter, not hard-coded | One-line swap between real and fake |
| **Bounded Retry** | Retry up to N times, then raise | Testable by checking `call_count` |
| **Output Parsing** | `json.loads()` on LLM response | Fake returns malformed strings to test error paths |

## 🎯 Summary

### Key Takeaways

1. **Test doubles replace real dependencies** — a simple `FakeAIClient` class mimics the OpenAI interface and returns controlled responses, making tests fast, free, and deterministic
2. **Dependency injection is the enabler** — passing the client as a parameter (instead of creating it inside the function) is the single change that makes offline testing possible
3. **Failure paths are the most valuable tests** — malformed JSON, timeouts, and exhausted retries are hard to trigger with a real API but trivial to simulate with a fake
4. **Bounded retry is verifiable** — by checking `fake.call_count`, you can assert that the retry loop ran exactly the right number of times